# Characterziation plots of the ```ZIP-load model```

In [ ]:
%load_ext autoreload
%autoreload 2

## 0 Imports and predefinitions

In [ ]:
# %matplotlib widget
# import ipympl

import os
import sys
import numpy as np
import pandas as pd

parent = os.path.abspath("../..")
sys.path.insert(1, parent)
sys.path.append(
    str(
        os.path.dirname(
            os.path.dirname(os.path.abspath("zip-load_characterization.ipynb"))
        )
    )
)

import matplotlib.pyplot as plt
import matplotlib as mpl
from diffpssi.tools import *

from diffpssi.power_sim_lib.models.static_models import Load
from diffpssi.power_sim_lib.backend import *

## 1 Creation of load element and characterization

In [ ]:
variation = [
    {"name": "constant Z Load", "bus": "Bus 0", "P": 2000, "Q": 200, "model": "Z"},
    {"name": "constant I Load", "bus": "Bus 0", "P": 2000, "Q": 200, "model": "I"},
    {"name": "constant P Load", "bus": "Bus 0", "P": 2000, "Q": 200, "model": "P"},
    # {'name': 'Load 4', 'bus': 'Bus 0', 'P': 2000, 'Q': 200, 'model': 'T'},
]

load_dict = None
zip_loads = []
for param_dict in variation:
    zip_loads.append(Load(s_n_sys=2200, param_dict=param_dict))

parallel_sims = 1
v_bb = torch.ones((parallel_sims, 1), dtype=torch.float64) * 1.0

for model in zip_loads:
    model.enable_parallel_simulation(parallel_sims=parallel_sims)
    model.initialize(s_calc=0, v_bb=v_bb)  # s_calc currently unused in calculation

Functions to be tested and characterized are:
1. ```get_lf_power()``` -> returning the apparent power for the load flow analysis
2. ```get_admittance()```
3. ```initialize()``` -> just setting the start-value of the load admittance 
4. ```calc_current_injections()``` -> returning shape like of zeros

But with respect to the state of the art implementation: ```calc_current_injections()``` is returning an empty tensor, meaning the loads are only represented through the admittance matrix. No current in- or resp. ejections take place.

In [ ]:
input = np.linspace(0.8, 1.5, 1000)
response = np.zeros((len(input), len(zip_loads)), dtype=complex)
real_power = np.zeros((len(input), len(zip_loads)), dtype=complex)

for j, model in enumerate(zip_loads):
    for i, v in enumerate(input):
        model.update_internal_vars(
            v_bb=v * torch.ones((parallel_sims, 1), dtype=torch.float64)
        )
        response[i][j] = model.get_admittance(dyn=True)
        real_power[i][j] = model.p_atm

## 2 Saving data, generation and saving of plots 

In [ ]:
# plotting the admittance of the load, dependent on the voltage
plt.figure()
for i, model in enumerate(zip_loads):
    plt.plot(input, response[:, i], label=model.name)

plt.legend()
plt.title(r"Characterization Impedance $Z$ vs. Voltage")
plt.xlabel("Voltage (p.u.)")
plt.ylabel(r"Impedance $Z$ (p.u.)")
plt.grid()
plt.savefig("./data/zip-load_characterization_impedance.pdf")

plt.show()

# plot of the power development of the load
plt.figure()
for i, model in enumerate(zip_loads):
    plt.plot(input, real_power[:, i], label=model.name)

plt.legend()
plt.title(r"Characterization Real Power $P$ vs. Voltage")
plt.xlabel("Voltage (p.u.)")
plt.ylabel(r"Power $P$ (MW)")  # is that really p.u.?
plt.grid()
plt.savefig("./data/zip-load_characterization_power.pdf")

plt.show()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

for i, model in enumerate(zip_loads):
    axs[0].plot(input, response[:, i], label=model.name)
    axs[1].plot(input, real_power[:, i], label=model.name)

# axs[0].set_xlabel('Voltage (p.u.)')
axs[0].set_ylabel(r"Impedance $Z$ (p.u.)")
axs[0].grid()


axs[1].set_xlabel("Voltage (p.u.)")
axs[1].set_ylabel(r"Power $P$ (MW)")  # is that really p.u.?
axs[1].grid()
plt.legend()
# fig.suptitle('Characterization of the ZIP Load model')
plt.savefig("./data/zip-load_characterization_full.pdf")
plt.tight_layout()
plt.show()

Exporting the admittance and real power data to a ```.csv``` file. If necessary, it can be further processed.

In [ ]:
data = pd.DataFrame(None)
data["Voltage"] = input
for i, model in enumerate(zip_loads):
    data[model.name] = response[:, i]
    data[model.name + " - real power P"] = real_power[:, i]
data.set_index("Voltage", inplace=True)

data.to_csv("./data/zip-load_characterization.csv")